<a href="https://colab.research.google.com/github/saitejamudapalli/Project-HealthCare-Provider-Analysis/blob/main/FACT_TABLES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#CODE FOR FACT TABLES


# ===========================================
# ✅ FACT_HEALTHCARE (Completely Clean — No NULLs or NaNs)
# ===========================================

# Create base fact table from the source DataFrame
fact_healthcare = pd.DataFrame()

fact_healthcare["patient_id"] = df["id"].astype(str)
fact_healthcare["time_id"] = range(1, len(df) + 1)
fact_healthcare["gender"] = df["gender"].astype(str)
fact_healthcare["marital_status"] = df["marital"].astype(str)
fact_healthcare["race"] = df["race"].astype(str)
fact_healthcare["ethnicity"] = df["ethnicity"].astype(str)
fact_healthcare["birthplace"] = df["birthplace"].astype(str)
fact_healthcare["city"] = df["city"].astype(str)
fact_healthcare["state"] = df["state"].astype(str)
fact_healthcare["county"] = df["county"].astype(str)
fact_healthcare["zip"] = df["zip"].astype(str)

# ✅ Convert "nan", "None", "null", "NaN", empty strings → real NaN
fact_healthcare = fact_healthcare.replace(
    to_replace=["nan", "NaN", "None", "NULL", "null", ""],
    value=pd.NA
)

# ✅ Drop ALL rows that contain ANY null / NaN value
fact_healthcare = fact_healthcare.dropna(how="any").reset_index(drop=True)

# ✅ Remove duplicates just in case
fact_healthcare = fact_healthcare.drop_duplicates()

# ✅ Recreate surrogate key
fact_healthcare["fact_id"] = range(1, len(fact_healthcare) + 1)

# --- BigQuery Load ---
table_id = f"{project_id}.{dataset_id}.fact_healthcare"
client.delete_table(table_id, not_found_ok=True)

schema = [
    bigquery.SchemaField("fact_id", "INTEGER"),
    bigquery.SchemaField("patient_id", "STRING"),
    bigquery.SchemaField("time_id", "INTEGER"),
    bigquery.SchemaField("gender", "STRING"),
    bigquery.SchemaField("marital_status", "STRING"),
    bigquery.SchemaField("race", "STRING"),
    bigquery.SchemaField("ethnicity", "STRING"),
    bigquery.SchemaField("birthplace", "STRING"),
    bigquery.SchemaField("city", "STRING"),
    bigquery.SchemaField("state", "STRING"),
    bigquery.SchemaField("county", "STRING"),
    bigquery.SchemaField("zip", "STRING"),
]

client.create_table(bigquery.Table(table_id, schema=schema))
client.load_table_from_dataframe(fact_healthcare, table_id).result()

print(f"✅ fact_healthcare created and loaded successfully — {len(fact_healthcare)} perfectly clean rows (no NaN / NULL).")
